# 03 — Model Experiments

**Purpose:** perform rolling temporal validation, tune CatBoost on a representative lifecycle configuration, train checkpoint/horizon model families, and evaluate untouched chronological holdouts.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)


Project root: C:\Users\BaneleS\Downloads\Game Performance Model


In [2]:
import inspect

from game_performance.config import SETTINGS
from game_performance.io import load_artifact, save_artifact
from game_performance.modeling import train_all_model_families

SETTINGS.ensure_directories()

historical_snapshots = load_artifact(SETTINGS.feature_dir / "historical_snapshots.joblib")
print("Loaded configurations:", len(historical_snapshots))

# Show what is available before production training starts.
for key, snapshot in historical_snapshots.items():
    checkpoint, horizon = key
    print(f"checkpoint={checkpoint:>2}, horizon={horizon:>2}, rows={len(snapshot):,}")

Loaded configurations: 1
checkpoint=14, horizon=14, rows=9,301


In [3]:
# Production training: no max_families limit, all targets, regression + classification.
# This cell is intentionally compatible with both the original modeling.py and the improved modeling.py.

signature = inspect.signature(train_all_model_families)
production_kwargs = {}

if "mode" in signature.parameters:
    production_kwargs["mode"] = "production"

if "max_families" in signature.parameters:
    production_kwargs["max_families"] = None

if "run_regression" in signature.parameters:
    production_kwargs["run_regression"] = True

if "run_classification" in signature.parameters:
    production_kwargs["run_classification"] = True

print("Production training kwargs:", production_kwargs or "default full-production call")

training_results = train_all_model_families(
    historical_snapshots,
    SETTINGS,
    **production_kwargs,
)

print("Trained model families:", len(training_results["model_bundles"]))


Production training kwargs: {'mode': 'production', 'max_families': None, 'run_regression': True, 'run_classification': True}
Training family day 14 -> horizon 14 [production] ...
Finished day 14 -> horizon 14 in 155.9s
Trained model families: 1


In [4]:
holdout = training_results.get("holdout_predictions")
skipped = training_results.get("skipped_configurations")

print("Regression evaluation:", training_results["regression_evaluation"].shape)
print("Classification evaluation:", training_results["classification_evaluation"].shape)
print("Holdout predictions:", holdout.shape if holdout is not None else None)

if holdout is not None and not holdout.empty:
    print("\nTrained holdout model families:")
    display(
        holdout.groupby(["Obs_Window", "Horizon"])
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["Obs_Window", "Horizon"])
    )

print("\nSkipped configurations:")
if skipped is not None and not skipped.empty:
    display(skipped)
else:
    print("None")


Regression evaluation: (2, 22)
Classification evaluation: (1, 31)
Holdout predictions: (498, 28)

Trained holdout model families:


,Obs_Window,Horizon,rows
0,14,14,498



Skipped configurations:
None


In [5]:
display(training_results["regression_evaluation"])
display(training_results["classification_evaluation"])

,Checkpoint,Horizon,Target,Target_Transform,Holdout_Start,Best_Iteration,_target_transform,depth,learning_rate,l2_leaf_reg,...,MAE,RMSE,R2,WAPE,Model_Blend_Weight,Baseline_MAE,Baseline_RMSE,Baseline_R2,Baseline_WAPE,WAPE_Improvement
0,14,14,target_clients,raw,2026-06-08,283,raw,7,0.03,8,...,3773.303449,3.728492e+04,0.500426,0.602587,0.8,3.929954e+03,2.040959e+04,0.850307,0.627604,0.025017
1,14,14,target_stakes,raw,2026-06-08,289,raw,7,0.03,8,...,743085.981493,3.300600e+06,0.684657,0.490114,0.9,1.225085e+06,5.185118e+06,0.221759,0.808024,0.317910


,Checkpoint,Horizon,Holdout_Start,Best_Iteration,Calibrator_Method,Classification_Threshold,Accuracy,Precision,Recall,F1,...,Validation_F1,Validation_ROC_AUC,Validation_Brier_Score,Validation_Log_Loss,Validation_Expected_Calibration_Error,depth,learning_rate,l2_leaf_reg,random_strength,bagging_temperature
0,14,14,2026-06-08,117,isotonic,0.5,0.909639,0.774436,0.872881,0.820717,...,0.858974,0.971254,0.059855,0.185042,1.595665e-17,5,0.05,5,1.0,0.5


In [6]:
reg = training_results["regression_evaluation"]

if not reg.empty:
    best_horizons = (
        reg.sort_values(["Checkpoint", "Target", "WAPE", "MAE"])
        .groupby(["Checkpoint", "Target"], as_index=False)
        .first()
    )
    display(best_horizons)
else:
    print("No regression evaluation rows were produced.")


,Checkpoint,Target,Horizon,Target_Transform,Holdout_Start,Best_Iteration,_target_transform,depth,learning_rate,l2_leaf_reg,...,MAE,RMSE,R2,WAPE,Model_Blend_Weight,Baseline_MAE,Baseline_RMSE,Baseline_R2,Baseline_WAPE,WAPE_Improvement
0,14,target_clients,14,raw,2026-06-08,283,raw,7,0.03,8,...,3773.303449,3.728492e+04,0.500426,0.602587,0.8,3.929954e+03,2.040959e+04,0.850307,0.627604,0.025017
1,14,target_stakes,14,raw,2026-06-08,289,raw,7,0.03,8,...,743085.981493,3.300600e+06,0.684657,0.490114,0.9,1.225085e+06,5.185118e+06,0.221759,0.808024,0.317910


In [7]:
save_artifact(training_results, SETTINGS.model_dir / "training_results.joblib")
print("Saved production training results:", SETTINGS.model_dir / "training_results.joblib")


Saved production training results: C:\Users\BaneleS\Downloads\Game Performance Model\artifacts\models\training_results.joblib
